# LeetCode 2217 — Find Palindrome With Fixed Length bằng Python

Link đề: https://leetcode.com/problems/find-palindrome-with-fixed-length/ · Độ khó: **Medium** · Chủ đề: Array, Math

Skeleton LeetCode:

```python
class Solution:
    def kthPalindrome(self, queries: List[int], intLength: int) -> List[int]:
        ...
```

## 1. Mục tiêu học

1. Nhận ra một số palindrome độ dài `L` được **xác định hoàn toàn** bởi **nửa đầu** gồm `h = ⌈L/2⌉` chữ số.
2. Thấy rằng thứ tự tăng dần của palindrome trùng với thứ tự tăng dần của nửa đầu ⇒ palindrome thứ `q` có nửa đầu `10^{h−1} + q − 1`.
3. Dựng palindrome từ nửa đầu bằng **chuỗi** (đảo ngược, bỏ chữ số giữa khi `L` lẻ) và bằng **số học** (lấy từng chữ số).
4. Kiểm tra **giới hạn**: chỉ có `9·10^{h−1}` palindrome độ dài `L`; truy vấn vượt quá trả `-1`.
5. Đối chiếu với liệt kê vét cạn cho `L` nhỏ, và kiểm tính chất (là palindrome, đúng độ dài, tăng dần) cho `L` lớn.

## 2. Đề bài & ví dụ

Cho mảng `queries` và số nguyên dương `intLength`. Với mỗi `q = queries[i]`, trả về palindrome **dương nhỏ thứ `q`** có **đúng** `intLength` chữ số, hoặc `-1` nếu không tồn tại. Palindrome **không** có số 0 đứng đầu.

**Ví dụ 1:** `queries = [1,2,3,4,5,90]`, `intLength = 3` → `[101,111,121,131,141,999]`. Các palindrome 3 chữ số: `101, 111, 121, …, 191, 202, …`; thứ 90 là `999`.

**Ví dụ 2:** `queries = [2,4,6]`, `intLength = 4` → `[1111,1331,1551]` (dãy: `1001, 1111, 1221, 1331, 1441, 1551, …`).

**Ràng buộc:** `1 ≤ len(queries) ≤ 5·10⁴`, `1 ≤ queries[i] ≤ 10⁹`, `1 ≤ intLength ≤ 15`.

In [1]:
import random
import time
from bisect import bisect_left
from typing import List

examples = [([1, 2, 3, 4, 5, 90], 3, [101, 111, 121, 131, 141, 999]), ([2, 4, 6], 4, [1111, 1331, 1551])]
for q, L, exp in examples:
    print(f"queries={q}, intLength={L} -> {exp}")

queries=[1, 2, 3, 4, 5, 90], intLength=3 -> [101, 111, 121, 131, 141, 999]
queries=[2, 4, 6], intLength=4 -> [1111, 1331, 1551]


## 3. Kiến thức nền

### 3.1 Kiểm tra palindrome và đảo chuỗi

`s == s[::-1]`. Với số: `str(x) == str(x)[::-1]`.

In [2]:
for x in (121, 1221, 123, 1001):
    print(x, str(x) == str(x)[::-1])

121 True
1221 True
123 False
1001 True


### 3.2 Nửa đầu và phép "phản chiếu"

Độ dài `L`, nửa đầu dài `h = (L + 1) // 2`. Palindrome = `half + reverse(half)` nếu `L` chẵn; `half + reverse(half)` **bỏ chữ số đầu của phần đảo** (chính là chữ số giữa) nếu `L` lẻ. Viết gọn: `s + s[::-1][L % 2:]`.

In [3]:
def mirror(half, L):
    s = str(half)
    return int(s + s[::-1][L % 2:])

print(mirror(12, 3), mirror(12, 4), mirror(100, 5), mirror(9, 1))

121 1221 10001 9


### 3.3 Số các nửa đầu hợp lệ

Nửa đầu có `h` chữ số, chữ số đầu khác 0 ⇒ nằm trong `[10^{h−1}, 10^h − 1]`, có `9·10^{h−1}` giá trị. Đó cũng là số palindrome độ dài `L`.

In [4]:
for L in range(1, 8):
    h = (L + 1) // 2
    print(f"L={L}: h={h}, nửa đầu từ {10 ** (h - 1)} tới {10 ** h - 1}, có {9 * 10 ** (h - 1)} palindrome")

L=1: h=1, nửa đầu từ 1 tới 9, có 9 palindrome
L=2: h=1, nửa đầu từ 1 tới 9, có 9 palindrome
L=3: h=2, nửa đầu từ 10 tới 99, có 90 palindrome
L=4: h=2, nửa đầu từ 10 tới 99, có 90 palindrome
L=5: h=3, nửa đầu từ 100 tới 999, có 900 palindrome
L=6: h=3, nửa đầu từ 100 tới 999, có 900 palindrome
L=7: h=4, nửa đầu từ 1000 tới 9999, có 9000 palindrome


### 3.4 Lấy chữ số bằng `divmod`

`x, d = divmod(x, 10)` tách chữ số cuối `d`. Dùng để phản chiếu bằng số học (không dùng chuỗi).

In [5]:
x, digits = 1234, []
while x:
    x, d = divmod(x, 10); digits.append(d)
print("chữ số từ phải sang trái:", digits)

chữ số từ phải sang trái: [4, 3, 2, 1]


### 3.5 So sánh số cùng độ dài = so sánh chuỗi

Với hai số **cùng số chữ số** (không có số 0 đứng đầu), `a < b` ⇔ `str(a) < str(b)` theo thứ tự từ điển. Đây là lý do "so sánh palindrome ⇔ so sánh nửa đầu": hai chuỗi cùng độ dài so sánh tại vị trí khác nhau **đầu tiên**, và vị trí đó luôn nằm trong nửa đầu (phần sau là phản chiếu).

In [6]:
pairs = [(1221, 1331), (12921, 13031), (99, 101)]
for a, b in pairs:
    same_len = len(str(a)) == len(str(b))
    print(a, b, "| a<b:", a < b, "| str(a)<str(b):", str(a) < str(b), "| cùng độ dài:", same_len)

1221 1331 | a<b: True | str(a)<str(b): True | cùng độ dài: True
12921 13031 | a<b: True | str(a)<str(b): True | cùng độ dài: True
99 101 | a<b: True | str(a)<str(b): False | cùng độ dài: False


## 4. Ý tưởng nền tảng / trực giác

Liệt kê palindrome 3 chữ số theo thứ tự tăng: `101, 111, 121, …, 191, 202, 212, …`. Nửa đầu (2 chữ số đầu) lần lượt là `10, 11, 12, …, 19, 20, 21, …` — **tăng đều từng đơn vị**, bắt đầu từ `10`.

Vì sao? So sánh hai palindrome cùng độ dài ⇔ so sánh nửa đầu (nửa đầu quyết định các chữ số **có trọng số lớn**; phần sau chỉ là phản chiếu). Vậy palindrome thứ `q` có nửa đầu là nửa đầu nhỏ thứ `q` = `10^{h−1} + (q − 1)`.

Tính tay: `L = 3`, `q = 90` ⇒ `h = 2`, nửa đầu `10 + 89 = 99` ⇒ `99 | 9` ⇒ `999` ✓. `L = 4`, `q = 6` ⇒ `h = 2`, nửa đầu `15` ⇒ `1551` ✓. `L = 3`, `q = 91` ⇒ nửa đầu `100` có 3 chữ số > `h` ⇒ `-1`.

In [7]:
L = 3
pals = [x for x in range(100, 1000) if str(x) == str(x)[::-1]]
print("10 palindrome đầu:", pals[:10], "| nửa đầu:", [x // 10 for x in pals[:10]], "| tổng:", len(pals))

10 palindrome đầu: [101, 111, 121, 131, 141, 151, 161, 171, 181, 191] | nửa đầu: [10, 11, 12, 13, 14, 15, 16, 17, 18, 19] | tổng: 90


Thêm một góc nhìn cho `L` chẵn và lẻ: bảng dưới in 6 palindrome đầu và 3 palindrome cuối của `L = 4` và `L = 5` cùng nửa đầu tương ứng — nửa đầu luôn chạy liên tục từ `10^{h−1}` tới `10^h − 1`.

In [8]:
for L in (4, 5):
    h = (L + 1) // 2
    halves = list(range(10 ** (h - 1), 10 ** h))
    first = [(x, mirror(x, L)) for x in halves[:6]]
    last = [(x, mirror(x, L)) for x in halves[-3:]]
    print(f"L={L} (h={h}): đầu {first} ... cuối {last}")

L=4 (h=2): đầu [(10, 1001), (11, 1111), (12, 1221), (13, 1331), (14, 1441), (15, 1551)] ... cuối [(97, 9779), (98, 9889), (99, 9999)]
L=5 (h=3): đầu [(100, 10001), (101, 10101), (102, 10201), (103, 10301), (104, 10401), (105, 10501)] ... cuối [(997, 99799), (998, 99899), (999, 99999)]


## 5. Cách 1 — Brute Force / baseline: duyệt mọi số độ dài `L`

**Ý tưởng:** duyệt `x` từ `10^{L−1}` tới `10^L − 1`, giữ danh sách palindrome, trả lời truy vấn bằng chỉ số. Chỉ khả thi với `L ≤ 6`.

In [9]:
def kth_brute(queries, L):
    pals = [x for x in range(10 ** (L - 1), 10 ** L) if str(x) == str(x)[::-1]]
    return [pals[q - 1] if q <= len(pals) else -1 for q in queries]

for q, L, exp in examples:
    got = kth_brute(q, L); print(got); assert got == exp

[101, 111, 121, 131, 141, 999]
[1111, 1331, 1551]


**Độ phức tạp:** `O(10^L · L)` — `L = 15` là `10¹⁵` số. Không thể.

**Nút thắt:** chỉ có `9·10^{h−1}` palindrome trong `9·10^{L−1}` số — ta duyệt quá nhiều số không phải palindrome.

In [10]:
for L in (4, 5, 6):
    t = time.perf_counter(); kth_brute([1], L)
    print(f"L={L}: {time.perf_counter() - t:.3f}s")

L=4: 0.001s
L=5: 0.009s
L=6: 0.089s


## 6. Ý tưởng thuật toán / bước đột phá

**Song ánh nửa đầu ↔ palindrome:** `half ∈ [10^{h−1}, 10^h − 1]` ↦ `mirror(half, L)` là song ánh **bảo toàn thứ tự** giữa nửa đầu `h` chữ số và palindrome `L` chữ số.

**Quy tắc "nếu… thì…":**
- `h = (L + 1) // 2`, `half = 10^{h−1} + q − 1`.
- **Nếu** `half ≥ 10^h` (tức `q > 9·10^{h−1}`) **thì** `-1`.
- **Ngược lại** trả `mirror(half, L)`.

**Bảo toàn thứ tự (vì sao):** nếu `half1 < half2` (cùng `h` chữ số) thì tại vị trí chữ số đầu tiên khác nhau `j < h`, palindrome cũng khác nhau lần đầu tại `j` với cùng quan hệ — các chữ số phía sau không ảnh hưởng.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Sinh mọi palindrome từ nửa đầu rồi tra cứu

Sinh `9·10^{h−1}` palindrome bằng `mirror` (theo thứ tự tăng) — nhanh hơn Cách 1 rất nhiều (`L = 10` chỉ `9·10⁴` số), nhưng `L = 15` thì `9·10⁷` — vẫn quá nhiều.

In [11]:
def kth_generate(queries, L):
    h = (L + 1) // 2
    pals = [mirror(half, L) for half in range(10 ** (h - 1), 10 ** h)]
    return [pals[q - 1] if q <= len(pals) else -1 for q in queries]

for q, L, exp in examples:
    assert kth_generate(q, L) == exp
print([kth_generate(q, L) for q, L, _ in examples])

[[101, 111, 121, 131, 141, 999], [1111, 1331, 1551]]


**Độ phức tạp Cách 2:** `O(9·10^{h−1} · L + |queries|)`.

### 7.2 Cách 3 — Công thức + phản chiếu bằng chuỗi (bản nộp chính)

In [12]:
def kth_formula(queries, L):
    h = (L + 1) // 2
    start, limit = 10 ** (h - 1), 10 ** h
    ans = []
    for q in queries:
        half = start + q - 1
        if half >= limit:
            ans.append(-1)
        else:
            s = str(half)
            ans.append(int(s + s[::-1][L % 2:]))
    return ans

for q, L, exp in examples:
    assert kth_formula(q, L) == exp
print([kth_formula(q, L) for q, L, _ in examples])

[[101, 111, 121, 131, 141, 999], [1111, 1331, 1551]]


**Độ phức tạp Cách 3:** `O(|queries| · L)` (mỗi truy vấn tạo chuỗi độ dài `L ≤ 15`). Space `O(|queries|)` cho đáp án.

### 7.3 Cách 4 — Phản chiếu bằng số học (không dùng chuỗi)

Lấy `half`, rồi "nối" các chữ số của `half` theo thứ tự ngược vào cuối: với `L` lẻ, bỏ chữ số cuối của `half` trước khi phản chiếu (đó là chữ số giữa).

In [13]:
def mirror_arith(half, L):
    res = half
    x = half // 10 if L % 2 else half          # L lẻ: không lặp lại chữ số giữa
    while x:
        x, d = divmod(x, 10)
        res = res * 10 + d
    return res

def kth_arith(queries, L):
    h = (L + 1) // 2
    start, limit = 10 ** (h - 1), 10 ** h
    return [mirror_arith(start + q - 1, L) if start + q - 1 < limit else -1 for q in queries]

for q, L, exp in examples:
    assert kth_arith(q, L) == exp
print([kth_arith(q, L) for q, L, _ in examples], "| L=1:", kth_arith([1, 9, 10], 1))

[[101, 111, 121, 131, 141, 999], [1111, 1331, 1551]] | L=1: [1, 9, -1]


**Độ phức tạp Cách 4:** `O(|queries| · L)`, không cấp phát chuỗi.

### 7.4 Cách 5 — Sinh palindrome theo từng chữ số (đếm tổ hợp)

Một cách nhìn "đếm": chữ số đầu có 9 lựa chọn (`1..9`), mỗi chữ số tiếp theo của nửa đầu có 10 lựa chọn. Palindrome thứ `q` (0-indexed `r = q − 1`) có nửa đầu là biểu diễn của `r` trong "hệ hỗn hợp" (9, 10, 10, …). Đây là cùng công thức nhưng dựng từng chữ số — hữu ích khi biến thể có ràng buộc riêng cho từng chữ số (ví dụ chỉ dùng chữ số lẻ).

In [14]:
def kth_digits(queries, L):
    h = (L + 1) // 2; total = 9 * 10 ** (h - 1); res = []
    for q in queries:
        if q > total: res.append(-1); continue
        r = q - 1; digits = []
        for pos in range(h - 1, -1, -1):          # từ chữ số có trọng số lớn nhất
            block = 10 ** pos
            d, r = divmod(r, block)
            digits.append(d + (1 if not digits else 0))   # chữ số đầu bắt đầu từ 1
        half = int("".join(map(str, digits)))
        res.append(mirror(half, L))
    return res

for q, L, exp in examples:
    assert kth_digits(q, L) == exp
print([kth_digits(q, L) for q, L, _ in examples])

[[101, 111, 121, 131, 141, 999], [1111, 1331, 1551]]


**Độ phức tạp Cách 5:** `O(|queries| · L)`.

## 8. TRACE từng bước

Trace Cách 3 trên ví dụ 1 (`L = 3` ⇒ `h = 2`, `start = 10`, `limit = 100`):

| q | half = 10 + q − 1 | half < 100? | s | s[::-1][1:] | palindrome |
|---|---|---|---|---|---|
| 1 | 10 | ✓ | "10" | "1" | 101 |
| 5 | 14 | ✓ | "14" | "1" | 141 |
| 90 | 99 | ✓ | "99" | "9" | 999 |
| 91 | 100 | ✗ | | | -1 |

In [15]:
def trace_formula(queries, L):
    h = (L + 1) // 2; start, limit = 10 ** (h - 1), 10 ** h
    print(f"L={L}: h={h}, nửa đầu hợp lệ trong [{start}, {limit - 1}]")
    out = []
    for q in queries:
        half = start + q - 1
        if half >= limit:
            print(f"  q={q}: half={half} >= {limit} -> -1"); out.append(-1); continue
        s = str(half); tail = s[::-1][L % 2:]
        print(f"  q={q}: half={half}, s={s!r}, phần phản chiếu={tail!r} -> {s + tail}")
        out.append(int(s + tail))
    return out

print(trace_formula([1, 5, 90, 91], 3))
print(trace_formula([1, 2, 6, 91], 4))

L=3: h=2, nửa đầu hợp lệ trong [10, 99]
  q=1: half=10, s='10', phần phản chiếu='1' -> 101
  q=5: half=14, s='14', phần phản chiếu='1' -> 141
  q=90: half=99, s='99', phần phản chiếu='9' -> 999
  q=91: half=100 >= 100 -> -1
[101, 141, 999, -1]
L=4: h=2, nửa đầu hợp lệ trong [10, 99]
  q=1: half=10, s='10', phần phản chiếu='01' -> 1001
  q=2: half=11, s='11', phần phản chiếu='11' -> 1111
  q=6: half=15, s='15', phần phản chiếu='51' -> 1551
  q=91: half=100 >= 100 -> -1
[1001, 1111, 1551, -1]


In [16]:
def trace_arith(half, L):
    res = half; x = half // 10 if L % 2 else half
    print(f"half={half}, L={L}: bắt đầu res={res}, phần cần đảo x={x}")
    while x:
        x, d = divmod(x, 10); res = res * 10 + d
        print(f"  lấy chữ số {d} -> res={res}, còn x={x}")
    return res

trace_arith(123, 5); trace_arith(123, 6)

half=123, L=5: bắt đầu res=123, phần cần đảo x=12
  lấy chữ số 2 -> res=1232, còn x=1
  lấy chữ số 1 -> res=12321, còn x=0
half=123, L=6: bắt đầu res=123, phần cần đảo x=123
  lấy chữ số 3 -> res=1233, còn x=12
  lấy chữ số 2 -> res=12332, còn x=1
  lấy chữ số 1 -> res=123321, còn x=0


123321

## 9. Code nộp LeetCode

In [17]:
class Solution:
    def kthPalindrome(self, queries: List[int], intLength: int) -> List[int]:
        h = (intLength + 1) // 2                  # số chữ số của nửa đầu
        start, limit = 10 ** (h - 1), 10 ** h
        odd = intLength % 2
        ans = []
        for q in queries:
            half = start + q - 1                  # nửa đầu nhỏ thứ q
            if half >= limit:
                ans.append(-1)                    # không đủ palindrome
            else:
                s = str(half)
                ans.append(int(s + s[::-1][odd:]))   # L lẻ: không lặp chữ số giữa
        return ans

for q, L, exp in examples:
    assert Solution().kthPalindrome(q, L) == exp
print("Ví dụ đề: OK")

Ví dụ đề: OK


## 10. Vì sao thuật toán đúng

**Song ánh.** Một palindrome `P` có `L` chữ số được xác định bởi `h = ⌈L/2⌉` chữ số đầu (các chữ số còn lại là phản chiếu). Chữ số đầu của `P` khác 0 ⇔ chữ số đầu của nửa đầu khác 0. Vậy `half ↦ mirror(half, L)` là song ánh từ `[10^{h−1}, 10^h − 1]` lên tập palindrome `L` chữ số.

**Bảo toàn thứ tự.** Với `half1 < half2` (cùng số chữ số), gọi `j` là chỉ số đầu tiên hai chuỗi khác nhau (`j < h`). Hai palindrome trùng nhau ở `j` chữ số đầu và khác nhau ở chữ số `j` với cùng chiều ⇒ `mirror(half1) < mirror(half2)` (cùng độ dài nên so sánh từ trái).

**Kết luận.** Palindrome nhỏ thứ `q` ứng với nửa đầu nhỏ thứ `q`, tức `10^{h−1} + q − 1`, tồn tại ⇔ `q ≤ 9·10^{h−1}`.

In [18]:
# Kiểm chứng song ánh + bảo toàn thứ tự cho mọi L <= 7
for L in range(1, 8):
    h = (L + 1) // 2
    gen = [mirror(x, L) for x in range(10 ** (h - 1), 10 ** h)]
    assert gen == sorted(gen) and len(set(gen)) == len(gen)
    assert all(len(str(p)) == L and str(p) == str(p)[::-1] for p in gen)
    if L <= 6:
        assert gen == [x for x in range(10 ** (L - 1), 10 ** L) if str(x) == str(x)[::-1]]
print("mirror là song ánh tăng dần lên tập palindrome (L <= 7), khớp liệt kê với L <= 6")

mirror là song ánh tăng dần lên tập palindrome (L <= 7), khớp liệt kê với L <= 6


## 11. Phân tích Time / Space

Gọi `Q = len(queries)`, `L = intLength ≤ 15`.

| Cách | Time | Space |
|---|---|---|
| 1. Duyệt mọi số | `O(10^L · L)` | `O(10^{L/2})` |
| 2. Sinh mọi palindrome | `O(10^{L/2} · L + Q)` | `O(10^{L/2})` |
| 3. Công thức + chuỗi | `O(Q · L)` | `O(Q)` |
| 4. Công thức + số học | `O(Q · L)` | `O(Q)` |
| 5. Từng chữ số | `O(Q · L)` | `O(Q)` |

Với `L = 15`, nửa đầu có 8 chữ số; đáp án tới `10¹⁵` — vượt `int32` (Java dùng `long`).

In [19]:
rng = random.Random(2217)
qs = [rng.randint(1, 10**9 if i % 10 == 0 else 9 * 10**7) for i in range(5 * 10**4)]   # 90% truy vấn hợp lệ với L = 15
for name, fn in [("formula", kth_formula), ("arith", kth_arith), ("digits", kth_digits),
                 ("Solution", lambda q, L: Solution().kthPalindrome(q, L))]:
    t = time.perf_counter(); r = fn(qs, 15)
    print(f"{name:9s}: {sum(1 for x in r if x != -1)} truy vấn có đáp án, ví dụ {r[:2]} ({time.perf_counter() - t:.3f}s)")

formula  : 45465 truy vấn có đáp án, ví dụ [-1, 438971262179834] (0.013s)


arith    : 45465 truy vấn có đáp án, ví dụ [-1, 438971262179834] (0.037s)
digits   : 45465 truy vấn có đáp án, ví dụ [-1, 438971262179834] (0.101s)
Solution : 45465 truy vấn có đáp án, ví dụ [-1, 438971262179834] (0.019s)


**Vì sao không "sinh sẵn" rồi tra cứu?** Cách 2 sinh `9·10^{h−1}` palindrome: với `L = 10` là `9·10⁴` (nhanh), `L = 12` là `9·10⁵` (vẫn được), nhưng `L = 15` là `9·10⁷` — vài chục giây và hàng GB bộ nhớ trong Python. Ô dưới đo Cách 2 với `L` tăng dần để thấy thời gian nhân 10 mỗi khi `h` tăng 1, trong khi Cách 3 gần như không đổi.

In [20]:
for L in (8, 10, 12):
    t = time.perf_counter(); kth_generate([1, 5, 77], L); t2 = time.perf_counter() - t
    t = time.perf_counter(); kth_formula([1, 5, 77], L); t3 = time.perf_counter() - t
    print(f"L={L}: sinh sẵn {t2:.4f}s | công thức {t3 * 1e6:.1f} µs")

L=8: sinh sẵn 0.0016s | công thức 2.9 µs
L=10: sinh sẵn 0.0171s | công thức 10.0 µs


L=12: sinh sẵn 0.1851s | công thức 10.4 µs


## 12. Bộ test

- Ví dụ đề.
- Biên: `L = 1` (`1..9`, truy vấn 10 ⇒ `-1`), `L = 2` (`11, 22, …, 99`), truy vấn đúng bằng số lượng tối đa, truy vấn vượt 1, `L = 15` với `q = 1` và `q = 9·10⁷`.
- Đối chiếu brute force mọi `L ≤ 6` với mọi `q ≤ số lượng + 2`; với `L` lớn kiểm tính chất (palindrome, độ dài, tăng dần theo `q`).

In [21]:
fns = {"brute": kth_brute, "generate": kth_generate, "formula": kth_formula, "arith": kth_arith,
       "digits": kth_digits, "Solution": lambda q, L: Solution().kthPalindrome(q, L)}
for q, L, exp in examples:
    for name, fn in fns.items():
        assert fn(q, L) == exp, name
assert Solution().kthPalindrome([1, 9, 10], 1) == [1, 9, -1]
assert Solution().kthPalindrome([1, 9, 10], 2) == [11, 99, -1]
assert Solution().kthPalindrome([900, 901], 5) == [99999, -1]
assert Solution().kthPalindrome([1, 9 * 10**7, 9 * 10**7 + 1], 15) == [100000000000001, 999999999999999, -1]
for L in range(1, 7):
    total = 9 * 10 ** ((L + 1) // 2 - 1)
    qs = list(range(1, total + 3))
    want = kth_brute(qs, L)
    for name, fn in fns.items():
        assert fn(qs, L) == want, (name, L)
print("Ví dụ + biên + vét cạn L <= 6: OK")

Ví dụ + biên + vét cạn L <= 6: OK


In [22]:
rng = random.Random(42)
for L in range(7, 16):
    qs = sorted(rng.randint(1, 9 * 10 ** ((L + 1) // 2 - 1)) for _ in range(200))
    out = Solution().kthPalindrome(qs, L)
    assert all(len(str(p)) == L and str(p) == str(p)[::-1] for p in out)
    assert all(out[i] <= out[i + 1] for i in range(len(out) - 1))
    assert out == kth_arith(qs, L) == kth_digits(qs, L)
print("L = 7..15: kết quả là palindrome đúng độ dài, tăng theo q, các cách khớp nhau")

L = 7..15: kết quả là palindrome đúng độ dài, tăng theo q, các cách khớp nhau


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Độ dài nửa đầu sai: `L // 2` thay vì `(L + 1) // 2`

In [23]:
def wrong_half_len(queries, L):
    h = L // 2                                           # SAI khi L lẻ
    start, limit = 10 ** max(h - 1, 0), 10 ** h
    out = []
    for q in queries:
        half = start + q - 1
        if half >= limit: out.append(-1); continue
        s = str(half); out.append(int(s + s[::-1][L % 2:]))
    return out

print("sai:", wrong_half_len([1, 2, 90], 3), "| đúng:", Solution().kthPalindrome([1, 2, 90], 3))

sai: [1, 2, -1] | đúng: [101, 111, 999]


In [24]:
print("sửa:", kth_formula([1, 2, 90], 3))

sửa: [101, 111, 999]


### Lỗi 2 — Lặp lại chữ số giữa khi `L` lẻ

In [25]:
def wrong_double_middle(queries, L):
    h = (L + 1) // 2; start, limit = 10 ** (h - 1), 10 ** h
    out = []
    for q in queries:
        half = start + q - 1
        if half >= limit: out.append(-1); continue
        s = str(half); out.append(int(s + s[::-1]))     # SAI: luôn đảo đủ
    return out

print("sai:", wrong_double_middle([1, 2], 3), "| đúng:", Solution().kthPalindrome([1, 2], 3))

sai: [1001, 1111] | đúng: [101, 111]


In [26]:
print("sửa (bỏ chữ số giữa: s[::-1][L % 2:]):", kth_formula([1, 2], 3))

sửa (bỏ chữ số giữa: s[::-1][L % 2:]): [101, 111]


### Lỗi 3 — Quên kiểm giới hạn (trả về palindrome dài hơn `L`)

In [27]:
def wrong_no_bound(queries, L):
    h = (L + 1) // 2; start = 10 ** (h - 1)
    out = []
    for q in queries:
        s = str(start + q - 1); out.append(int(s + s[::-1][L % 2:]))   # SAI: không kiểm half < 10^h
    return out

print("sai:", wrong_no_bound([90, 91], 3), "| đúng:", Solution().kthPalindrome([90, 91], 3))

sai: [999, 10001] | đúng: [999, -1]


In [28]:
print("sửa:", kth_formula([90, 91], 3))

sửa: [999, -1]


### Lỗi 4 — Cho nửa đầu bắt đầu từ `q − 1` (có số 0 đứng đầu)

Nửa đầu phải bắt đầu từ `10^{h−1}`, nếu không sẽ sinh "palindrome" có số 0 đầu (thực chất ngắn hơn).

In [29]:
def wrong_leading_zero(queries, L):
    h = (L + 1) // 2
    out = []
    for q in queries:
        s = str(q - 1).zfill(h)                           # SAI: cho phép số 0 đứng đầu
        out.append(int(s + s[::-1][L % 2:]))
    return out

print("sai:", wrong_leading_zero([1, 2, 11], 3), "| đúng:", Solution().kthPalindrome([1, 2, 11], 3))

sai: [0, 10, 101] | đúng: [101, 111, 202]


In [30]:
print("sửa (start = 10^(h-1)):", kth_formula([1, 2, 11], 3))

sửa (start = 10^(h-1)): [101, 111, 202]


### Lỗi 5 — Phản chiếu số học với `L` lẻ mà không bỏ chữ số giữa

In [31]:
def wrong_mirror_arith(half, L):
    res, x = half, half                                    # SAI: L lẻ phải dùng half // 10
    while x:
        x, d = divmod(x, 10); res = res * 10 + d
    return res

print("sai:", wrong_mirror_arith(12, 3), "| đúng:", mirror_arith(12, 3))

sai: 1221 | đúng: 121


In [32]:
print("sửa:", mirror_arith(12, 3), mirror_arith(12, 4))

sửa: 121 1221


### Lỗi 6 — Ghép nửa sau bằng số học nhưng làm mất số 0 của phần đảo ngược

`int(str(half)[::-1])` bỏ số 0 đứng đầu của phần đảo (ví dụ `half = 10` ⇒ đảo `"01"` ⇒ `1`). Nếu ghép bằng `half * 10**len(str(rev)) + rev` thì độ dài bị hụt: `L = 4`, `q = 1` ra `101` thay vì `1001`.

In [33]:
def wrong_lost_zero(queries, L):
    h = (L + 1) // 2; start, limit = 10 ** (h - 1), 10 ** h; out = []
    for q in queries:
        half = start + q - 1
        if half >= limit: out.append(-1); continue
        tail = str(half)[::-1][L % 2:]
        rev = int(tail) if tail else 0
        out.append(half * 10 ** len(str(rev)) + rev if tail else half)   # SAI: len(str(rev)) < len(tail)
    return out

print("sai:", wrong_lost_zero([1, 2, 10], 4), "| đúng:", Solution().kthPalindrome([1, 2, 10], 4))

sai: [101, 1111, 1991] | đúng: [1001, 1111, 1991]


In [34]:
print("sửa (nhân 10**len(tail), hoặc ghép chuỗi):", kth_formula([1, 2, 10], 4))

sửa (nhân 10**len(tail), hoặc ghép chuỗi): [1001, 1111, 1991]


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **Palindrome** ⇔ xác định bởi **nửa đầu**; đếm/liệt kê/tìm thứ `k` đều quy về nửa đầu.
- **"Phần tử thứ k trong một tập có cấu trúc"** ⇒ tìm song ánh bảo toàn thứ tự với một dãy đơn giản (ở đây `10^{h−1}, 10^{h−1} + 1, …`).
- `s + s[::-1][L % 2:]` là mẹo phản chiếu gọn cho cả `L` chẵn lẫn lẻ.
- Luôn kiểm **giới hạn số lượng** (`9·10^{h−1}`).
- Với `L` nhỏ, liệt kê vét cạn để kiểm công thức.

## 15. Biến thể của bài

1. **Thứ hạng (index) của một palindrome cho trước** (bài tập 5): `rank = half − 10^{h−1} + 1`.
2. **Palindrome nhỏ nhất ≥ x** (LeetCode 564 — Find the Closest Palindrome, Hard): thử phản chiếu nửa đầu của `x`, `±1`, và các trường hợp `99…9`, `10…01`.
3. **Palindrome thứ `k` với độ dài tùy ý** (đếm qua mọi độ dài): cộng dồn `9·10^{⌈L/2⌉−1}` theo `L`.
4. **LeetCode 906 — Super Palindromes (Hard):** sinh palindrome từ nửa đầu rồi bình phương.
5. **Chỉ dùng chữ số lẻ** (`1, 3, 5, 7, 9`): nửa đầu là biểu diễn cơ số 5 — dùng ý tưởng Cách 5.

Minh họa biến thể 3 (tự dựng, kiểm bằng liệt kê):

In [35]:
def kth_palindrome_any_length(k):
    L = 1
    while True:
        cnt = 9 * 10 ** ((L + 1) // 2 - 1)
        if k <= cnt:
            return Solution().kthPalindrome([k], L)[0]
        k -= cnt; L += 1

all_pals = [x for x in range(1, 10**6) if str(x) == str(x)[::-1]]
for k in [1, 9, 10, 18, 19, 108, 109, 1000, len(all_pals)]:
    assert kth_palindrome_any_length(k) == all_pals[k - 1]
print("Palindrome thứ k mọi độ dài khớp liệt kê; k=10 ->", kth_palindrome_any_length(10), "| k=1000 ->", kth_palindrome_any_length(1000))

Palindrome thứ k mọi độ dài khớp liệt kê; k=10 -> 11 | k=1000 -> 90109


Minh họa biến thể 2 (tự dựng) — **palindrome nhỏ nhất `≥ x`**. Ý tưởng: lấy nửa đầu của `x` (cùng độ dài `L`), phản chiếu; nếu kết quả `< x` thì tăng nửa đầu thêm 1 rồi phản chiếu lại; nếu nửa đầu tràn (`99…9 + 1`) thì đáp án là palindrome nhỏ nhất độ dài `L + 1` (`10…01`). Kiểm bằng duyệt tăng dần.

In [36]:
def next_palindrome_ge(x):
    s = str(x); L = len(s); h = (L + 1) // 2
    half = int(s[:h])
    cand = mirror(half, L)
    if cand >= x:
        return cand
    if half + 1 < 10 ** h:
        return mirror(half + 1, L)
    return 10 ** L + 1                         # 99...9 -> 100...001

def next_palindrome_brute(x):
    while str(x) != str(x)[::-1]:
        x += 1
    return x

rng = random.Random(564)
for x in list(range(1, 3000)) + [rng.randint(1, 10**7) for _ in range(300)]:
    assert next_palindrome_ge(x) == next_palindrome_brute(x)
print("Palindrome nhỏ nhất >= x khớp duyệt tăng dần; 12345 ->", next_palindrome_ge(12345), "| 999 ->", next_palindrome_ge(999))

Palindrome nhỏ nhất >= x khớp duyệt tăng dần; 12345 -> 12421 | 999 -> 999


Minh họa biến thể 5 (tự dựng) — **palindrome thứ `k` độ dài `L` chỉ dùng chữ số lẻ** `{1,3,5,7,9}`. Nửa đầu là biểu diễn **cơ số 5** của `k − 1` (đủ `h` chữ số), ánh xạ chữ số `d ↦ 2d + 1`. Không còn vấn đề số 0 đứng đầu vì mọi chữ số đều khác 0.

In [37]:
def kth_odd_digit_palindrome(k, L):
    h = (L + 1) // 2
    if k > 5 ** h: return -1
    r = k - 1; digits = []
    for _ in range(h):
        r, d = divmod(r, 5); digits.append(str(2 * d + 1))
    s = "".join(reversed(digits))
    return int(s + s[::-1][L % 2:])

for L in range(1, 6):
    brute = [x for x in range(10 ** (L - 1), 10 ** L) if str(x) == str(x)[::-1] and set(str(x)) <= set("13579")]
    assert [kth_odd_digit_palindrome(k, L) for k in range(1, len(brute) + 2)] == brute + [-1]
print("Biến thể chữ số lẻ khớp liệt kê (L <= 5); k=7, L=4 ->", kth_odd_digit_palindrome(7, 4))

Biến thể chữ số lẻ khớp liệt kê (L <= 5); k=7, L=4 -> 3333


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Duyệt mọi số | lọc palindrome | `O(10^L·L)` | lớn | kiểm tra, `L ≤ 6` |
| 2. Sinh từ nửa đầu | liệt kê `9·10^{h−1}` | `O(10^{L/2}·L)` | lớn | `L ≤ 10` |
| 3. Công thức + chuỗi | `half = 10^{h−1}+q−1` | `O(Q·L)` | `O(Q)` | **nộp bài** |
| 4. Công thức + số học | phản chiếu bằng `divmod` | `O(Q·L)` | `O(Q)` | tránh chuỗi |
| 5. Từng chữ số | hệ hỗn hợp (9,10,…) | `O(Q·L)` | `O(Q)` | biến thể ràng buộc chữ số |

## 17. Bài tập (từ dễ đến khó)

1. **Đoán output:** `kthPalindrome([1, 10, 100], 2)`.
2. **Tính tay:** `kthPalindrome([37], 5)`.
3. **Điền TODO:** `count_palindromes(L)` — số palindrome có đúng `L` chữ số.
4. **Tự viết lại:** viết lại Cách 3 không nhìn code.
5. **Mini challenge:** `palindrome_rank(p)` — cho palindrome dương `p`, trả về `q` sao cho `p` là palindrome thứ `q` trong các palindrome cùng độ dài. Kiểm ngược với `kthPalindrome`.

In [38]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def count_palindromes(L):
    # TODO
    pass

In [39]:
# Bài 4 — TODO (chỉ định nghĩa, không gọi)
def my_kth_palindrome(queries, L):
    # TODO
    pass

In [40]:
# Bài 5 — TODO (chỉ định nghĩa, không gọi)
def palindrome_rank(p):
    # TODO: lấy nửa đầu của str(p)
    pass

## 18. Gợi ý 3 mức

**Bài 1.** (1) `L = 2` ⇒ `h = 1`, nửa đầu `1..9`. (2) `q = 1 → 11`. (3) `q = 10, 100` vượt 9 ⇒ `-1`.

**Bài 2.** (1) `h = 3`, nửa đầu `100 + 36 = 136`. (2) `L` lẻ ⇒ phản chiếu bỏ chữ số giữa: `"136" + "31"`. (3) `13631`.

**Bài 3.** (1) `h = (L + 1) // 2`. (2) Chữ số đầu 9 lựa chọn, còn lại 10. (3) `9 · 10^{h−1}`.

**Bài 4.** (1) Tính `h`, `start`, `limit`. (2) `half = start + q − 1`. (3) Phản chiếu hoặc `-1`.

**Bài 5.** (1) `s = str(p)`, `h = (len(s) + 1) // 2`. (2) `half = int(s[:h])`. (3) `q = half − 10^{h−1} + 1`.

## 19. Đáp án tham khảo

In [41]:
# Bài 1, 2
assert Solution().kthPalindrome([1, 10, 100], 2) == [11, -1, -1]
assert Solution().kthPalindrome([37], 5) == [13631]
print("Bài 1: [11, -1, -1] | Bài 2: 13631")
trace_formula([37], 5)

Bài 1: [11, -1, -1] | Bài 2: 13631
L=5: h=3, nửa đầu hợp lệ trong [100, 999]
  q=37: half=136, s='136', phần phản chiếu='31' -> 13631


[13631]

In [42]:
# Bài 3, 4
def count_palindromes_ans(L):
    return 9 * 10 ** ((L + 1) // 2 - 1)

my_kth_palindrome_ans = kth_formula
for L in range(1, 7):
    assert count_palindromes_ans(L) == sum(1 for x in range(10 ** (L - 1), 10 ** L) if str(x) == str(x)[::-1])
    qs = list(range(1, count_palindromes_ans(L) + 2))
    assert my_kth_palindrome_ans(qs, L) == kth_brute(qs, L)
print("Bài 3, 4: OK;", [count_palindromes_ans(L) for L in range(1, 16)])

Bài 3, 4: OK; [9, 9, 90, 90, 900, 900, 9000, 9000, 90000, 90000, 900000, 900000, 9000000, 9000000, 90000000]


In [43]:
# Bài 5
def palindrome_rank_ans(p):
    s = str(p); h = (len(s) + 1) // 2
    return int(s[:h]) - 10 ** (h - 1) + 1

rng = random.Random(5)
for _ in range(2000):
    L = rng.randint(1, 15); q = rng.randint(1, count_palindromes_ans(L))
    p = Solution().kthPalindrome([q], L)[0]
    assert palindrome_rank_ans(p) == q
print("Bài 5: OK; rank(999) =", palindrome_rank_ans(999), "| rank(1001) =", palindrome_rank_ans(1001))

Bài 5: OK; rank(999) = 90 | rank(1001) = 1


**Giải thích thêm cho Bài 5.** `palindrome_rank` là hàm **ngược** của `kthPalindrome`: từ `p` lấy lại nửa đầu `half = int(s[:h])`, rồi `q = half − 10^{h−1} + 1`. Cặp hàm "thứ `k` ↔ thứ hạng" là công cụ mạnh: ví dụ đếm số palindrome độ dài `L` trong đoạn `[a, b]` = `rank(prev ≤ b) − rank(next ≥ a) + 1`. Ô dưới dùng `next_palindrome_ge` (mục 15) và `palindrome_rank_ans` để đếm palindrome trong một đoạn và so với đếm trực tiếp.

In [44]:
def count_palindromes_in_range(a, b):
    # chỉ xét a, b cùng số chữ số
    lo = next_palindrome_ge(a)
    if lo > b or len(str(lo)) != len(str(a)): return 0
    cnt = 0
    # palindrome lớn nhất <= b: duyệt giảm từ nửa đầu của b
    s = str(b); L = len(s); h = (L + 1) // 2; half = int(s[:h])
    hi = mirror(half, L)
    if hi > b: hi = mirror(half - 1, L) if half - 1 >= 10 ** (h - 1) else -1
    if hi == -1 or hi < lo: return 0
    return palindrome_rank_ans(hi) - palindrome_rank_ans(lo) + 1

rng = random.Random(9)
for _ in range(300):
    L = rng.randint(1, 5); a = rng.randint(10 ** (L - 1), 10 ** L - 1); b = rng.randint(a, 10 ** L - 1)
    assert count_palindromes_in_range(a, b) == sum(1 for x in range(a, b + 1) if str(x) == str(x)[::-1])
print("Đếm palindrome trong đoạn qua rank khớp đếm trực tiếp; [1000, 5000] ->", count_palindromes_in_range(1000, 5000))

Đếm palindrome trong đoạn qua rank khớp đếm trực tiếp; [1000, 5000] -> 40


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template:**

```python
h = (L + 1) // 2
start, limit = 10 ** (h - 1), 10 ** h
half = start + q - 1
if half >= limit: return -1
s = str(half)
return int(s + s[::-1][L % 2:])
```

In [45]:
def template(q, L):
    h = (L + 1) // 2; start, limit = 10 ** (h - 1), 10 ** h
    half = start + q - 1
    if half >= limit: return -1
    s = str(half); return int(s + s[::-1][L % 2:])
print([[template(q, L) for q in qs] for qs, L, _ in examples])

[[101, 111, 121, 131, 141, 999], [1111, 1331, 1551]]


**Checklist:**

- [ ] `h = (L + 1) // 2` (nửa đầu gồm cả chữ số giữa khi `L` lẻ).
- [ ] Nửa đầu bắt đầu từ `10^{h−1}` (không có số 0 đứng đầu).
- [ ] Kiểm `half < 10^h` (tối đa `9·10^{h−1}` palindrome).
- [ ] `L` lẻ: phản chiếu **bỏ** chữ số giữa (`s[::-1][1:]`).
- [ ] Đáp án tới `10¹⁵` — dùng `long` nếu dịch sang Java/C++.
- [ ] Đã thử `L = 1`, `L = 2`, truy vấn đúng bằng số lượng tối đa.